# Mini-Project 2: Solar Micro-Grid Dispatch Planner
**Scenario.** A rural health centre in Kasese must meet two daily constraints with solar (x) and battery (y) energy:

- 3x + 2y = D1 (daytime load, kWh)
- 4x + y = D2 (critical-equipment load, kWh)

All reusable logic lives in `src/microgrid.py` (classes `MicroGrid`, `HybridMicroGrid`); this notebook imports it and interprets the results. Tests are in `tests/test_microgrid.py`.

In [ ]:
import timeit
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.microgrid import (
    HybridMicroGrid, MicroGrid, SingularSystemError,
    generate_demand_csv, load_demand_csv, plot_dispatch, read_positive_float,
)

SEED = 42
rng = np.random.default_rng(SEED)   # used wherever randomness appears in this notebook
Path("figures").mkdir(exist_ok=True)
grid = MicroGrid()                  # A = [[3, 2], [4, 1]], solar UGX150/kWh, battery UGX450/kWh
grid

## Task 1: Is the system well-posed?
Before solving we check the determinant and the condition number.

In [ ]:
print("A =\n", grid.A)
print(f"det(A)            = {grid.determinant:.4f}")
print(f"cond(A)           = {grid.condition_number:.3f}")
sv = np.linalg.svd(grid.A, compute_uv=False)
print(f"singular values   = {sv.round(3)}  ->  cond = {sv[0]:.3f} / {sv[1]:.3f}")
print("hand check det    =", 3*1 - 2*4)
print("\nsolve_day(60, 55) =", grid.solve_day(60, 55), "(hand calc: x = (2*55-60)/5 = 10, y = (4*60-3*55)/5 = 15)")

**Interpretation.**
- **Determinant = -5 ≠ 0.** The two equations are not proportional (3:2 vs 4:1), so for any (D1, D2) there is exactly one (x, y). A determinant of 0 would mean dependent equations: no solution or infinitely many.
- **Condition number ≈ 5.8** (ratio of the largest to the smallest singular value). It bounds how much a *relative* error in the demands can be magnified in the solution: a 1% metering error could become at most ≈ 5.8% in (x, y). Values near 1 are ideal; values like 10⁸ or more mean the answer is numerically meaningless. Here the system is well conditioned, though not perfectly so.
- The determinant alone is scale-dependent (multiply a row by 1000 and it changes), which is why the condition number is the better health check. `MicroGrid` rejects a matrix with det ≈ 0 or a huge condition number by raising `SingularSystemError`.

## Task 2: Two input modes
**(a) Interactive.** `read_positive_float` re-prompts on empty, non-numeric, NaN/inf and negative input. To keep *Restart & Run All* working, the notebook uses scripted answers by default. Set `RUN_INTERACTIVE = True` to type values yourself.

In [ ]:
RUN_INTERACTIVE = False
if RUN_INTERACTIVE:
    d1 = read_positive_float("Daytime load D1 (kWh): ")
    d2 = read_positive_float("Critical load D2 (kWh): ")
else:
    scripted = iter(["", "abc", "-5", "64.9"])        # three bad answers, then a good one
    d1 = read_positive_float("Daytime load D1 (kWh): ", input_fn=lambda prompt: next(scripted))
    d2 = 55.0
print("Dispatch [solar, battery] =", grid.solve_day(d1, d2).round(2))

**(b) CSV with 30 days of demand**, generated with a seeded RNG: a weekly pattern (weekdays ≈ 70 kWh, Saturday 80% of that, Sunday 60%) plus Gaussian noise for the daytime load; a roughly constant ≈ 55 kWh critical load.

In [ ]:
csv_path = generate_demand_csv("data/demand_30days.csv", n_days=30, seed=SEED)
dates, D = load_demand_csv(csv_path)          # D has shape (2, 30)
print(D.shape, dates[0], "->", dates[-1])
print(Path(csv_path).read_text().splitlines()[:4])

## Task 3: Loop versus a single vectorised call

In [ ]:
X_loop = grid.solve_loop(D)
X_vec = grid.solve_vectorised(D)               # scipy.linalg.solve(A, D) with D of shape 2 x 30
assert np.allclose(X_loop, X_vec)

n = 1000
t_loop = timeit.timeit(lambda: grid.solve_loop(D), number=n) / n
t_vec = timeit.timeit(lambda: grid.solve_vectorised(D), number=n) / n
print(f"loop       : {t_loop*1e6:8.1f} microseconds per 30 days")
print(f"vectorised : {t_vec*1e6:8.1f} microseconds per 30 days")
print(f"speed-up   : {t_loop/t_vec:.1f}x")

**Comment.** Both give identical answers. The vectorised call is much faster because the matrix A is factorised **once** (LU) and reused for all 30 right-hand sides inside compiled LAPACK code, whereas the loop pays Python overhead, input checking and a fresh factorisation 30 times. For 30 days the absolute difference is trivial; it matters when scaling to many sites or thousands of Monte Carlo draws.

## Task 4: Infeasible days (negative x or y)

In [ ]:
plan_nnls = grid.plan(D, strategy="nnls")
plan_clip = grid.plan(D, strategy="clip")
bad = np.nonzero(plan_nnls.infeasible)[0]
print(f"{len(bad)} infeasible day(s) out of {D.shape[1]}\n")
print(f"{'date':<12}{'D1':>6}{'D2':>6}{'raw x':>8}{'raw y':>8} | {'NNLS x':>7}{'y':>6}{'unmet':>7} | {'clip x':>7}{'y':>6}{'unmet':>7}")
for j in bad:
    print(f"{dates[j]:<12}{D[0,j]:6.1f}{D[1,j]:6.1f}{plan_nnls.raw[0,j]:8.2f}{plan_nnls.raw[1,j]:8.2f} | "
          f"{plan_nnls.dispatch[0,j]:7.2f}{plan_nnls.dispatch[1,j]:6.2f}{plan_nnls.unmet[j]:7.2f} | "
          f"{plan_clip.dispatch[0,j]:7.2f}{plan_clip.dispatch[1,j]:6.2f}{plan_clip.unmet[j]:7.2f}")
weekday = [__import__("datetime").date.fromisoformat(dates[j]).strftime("%A") for j in bad]
print("\nInfeasible days fall on:", weekday)

**Why they occur.** Solving the system gives y = (4·D1 − 3·D2)/5, which is negative when D2 > (4/3)·D1. On quiet days the daytime load drops while the critical load stays constant, so the equations would need a *negative* battery draw (i.e. the battery would have to absorb energy).

**Strategy.** Both strategies leave feasible days untouched and report the **unmet demand** (2-norm of A·x − d). *Clipping* is simple, but the unmet energy is whatever happens to fall out. *NNLS* (`scipy.optimize.nnls`) finds the non-negative dispatch with the **smallest** unmet demand, so it is the better repair, and the comparison above shows it.  We use NNLS as the working plan.

## Task 5: Volatility of solar and battery usage

In [ ]:
X = plan_nnls.dispatch
stats = grid.usage_stats(X)
print(f"{'source':<9}{'mean (kWh)':>11}{'variance (kWh²)':>17}{'st.dev (kWh)':>14}{'CV':>7}")
for name, s in stats.items():
    print(f"{name:<9}{s['mean']:11.2f}{s['variance']:17.2f}{s['stdev']:14.2f}{s['cv']:7.2f}")
more = max(stats, key=lambda k: stats[k]["cv"])
print(f"\nMore volatile source (by CV): {more}")

Both variance and the unit-free coefficient of variation (CV = st.dev / mean) point the same way, so the conclusion does not depend on the two sources having different mean sizes.

## Task 6: Cost model (solar UGX 150/kWh, battery UGX 450/kWh)

In [ ]:
daily_cost = grid.daily_cost(X)
print(f"Mean daily cost : UGX {daily_cost.mean():,.0f}")
print(f"Cheapest / dearest day : UGX {daily_cost.min():,.0f} / UGX {daily_cost.max():,.0f}")
print(f"Monthly cost    : UGX {grid.monthly_cost(X):,.0f}")
solar_cost = grid.prices[0] * X[0].sum()
print(f"Solar supplies {X[0].sum()/X.sum():.0%} of the energy but only {solar_cost/daily_cost.sum():.0%} of the cost")

## Task 7: Stacked dispatch and daily cost

In [ ]:
fig = plot_dispatch(X, daily_cost, grid.labels, day_labels=dates)
fig.savefig("figures/project2_dispatch.png", dpi=150)
plt.show()

The chart supports the main point: battery energy (and hence cost) swings far more than solar, and the cheapest days are the quiet Sundays.

## Extension A: `HybridMicroGrid` (solar + battery + diesel)
The subclass only supplies a 3 x 3 matrix, a price vector and labels; all solving, validation and costing is inherited. Third equation (illustrative): x + y + z = D3, total energy balance.

In [ ]:
hybrid = HybridMicroGrid()
print(hybrid)
print("solve_day(100, 90, 40) ->", hybrid.solve_day(100, 90, 40).round(2), "[solar, battery, diesel]")

# Third row = row 1 + row 2  ->  linearly dependent
try:
    HybridMicroGrid(A=[[3, 2, 1], [4, 1, 2], [7, 3, 3]])
except SingularSystemError as err:
    print("Dependent system rejected:", err)

**If the new equation is linearly dependent**, det(A) = 0 (numerically ≈ 1e-15) and the condition number explodes (≈ 1e16). Then either no x, y, z satisfies all three equations (inconsistent demands) or infinitely many do (a free choice of generator output). `HybridMicroGrid` refuses to build such a system instead of returning an arbitrary answer. The cure is a genuinely new constraint, or an optimisation objective (e.g. minimise diesel) rather than a pure solve. The solution above also shows diesel can come out negative: the same feasibility issue as Task 4.

## Extension B: Sensitivity to ±5% demand errors (Monte Carlo, 1,000 draws)

In [ ]:
mean_demand = D.mean(axis=1)
sens = grid.sensitivity(mean_demand, rel=0.05, n_draws=1000, seed=SEED)
print("Baseline [solar, battery]      :", sens["baseline"].round(2))
print("Std of dispatch across draws   :", sens["std"].round(2))
print(f"Worst relative input change    : {sens['max_rel_input_change']:.1%}")
print(f"Worst relative output change   : {sens['max_rel_output_change']:.1%}")
print(f"Largest observed amplification : {sens['max_amplification']:.2f}x   (bound = cond(A) = {sens['condition_number']:.2f})")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(*sens["draws"], s=6, alpha=0.4)
ax.scatter(*sens["baseline"], color="red", label="baseline")
ax.set_xlabel("Solar x (kWh)"); ax.set_ylabel("Battery y (kWh)")
ax.set_title("1,000 draws with demands perturbed by ±5%"); ax.legend()
fig.tight_layout(); fig.savefig("figures/project2_sensitivity.png", dpi=150); plt.show()

A ±5% error in both demands moves the dispatch by up to roughly 20% in the worst draw. That is a magnification of about 4.4x, below the theoretical bound of cond(A) ≈ 5.8. The bound is an upper limit that only a particular worst-case direction of error reaches, so the observed amplification stays below it.

## Findings & Limitations
**Findings.** The system is well-posed (det = −5, cond ≈ 5.8), but "well-conditioned" still means that a ±5% metering error in demand moved the dispatch by up to ~21% in the Monte Carlo run (observed amplification 4.4× against a bound of 5.8×). Planners should therefore treat the dispatch as an estimate, not an exact schedule. The vectorised solve was ~28× faster than the loop because one factorisation of A serves all 30 right-hand sides; this is irrelevant for one month but matters for many sites or simulation. Two days (the Sundays 20 and 27 September) were infeasible: daytime load falls while the critical load stays constant, which would require the battery to absorb energy. NNLS leaves 3–4 kWh unmet versus 7–9 kWh for clipping, so it is the better repair. The battery is clearly the more volatile source (CV 0.45 against 0.28) and, at three times the price, drives about 87% of the monthly cost of roughly UGX 299,000, even though it supplies 68% of the energy.

**Limitations.** Demand is synthetic. Exact equality constraints force a specific energy mix rather than optimising one; a linear programme with inequalities, solar availability by hour and battery state of charge would be more realistic. A negative battery term could legitimately mean charging from surplus solar, which the model treats as infeasible. Prices are illustrative, one month cannot show seasonality, and a 30-point sample gives rough variance estimates.